| Feature                | FAISS   | ChromaDB |
| ---------------------- | ------- | -------- |
| Flat index             | ✅       | ❌        |
| IVF                    | ✅       | ❌        |
| HNSW                   | ✅       | ✅        |
| PQ                     | ✅       | ❌        |
| Manual index selection | ✅       | ❌        |
| Metadata filtering     | Limited | ✅        |
| Persistence            | Manual  | Built-in |
| LangChain integration  | ✅       | ✅        |


Final understanding
FAISS = vector index/search engine

You manually manage:
- index type
- dimension
- metric
- docstore
- ID mapping
- persistence
  
Chroma = vector database
It manages:
- vectors
- documents
- metadata
- IDs
- collections
- index
- persistence


FAISS is a low-level, high-performance library for dense-vector similarity search and clustering. It gives developers direct control over index structures such as Flat, IVF, HNSW and product-quantized indexes, and it offers strong CPU and GPU capabilities. However, FAISS is not a complete vector database: document storage, metadata management, filtering, CRUD APIs, collections, persistence orchestration and server infrastructure generally need to be handled separately.

Chroma is a retrieval database/search infrastructure designed for AI applications. It stores embeddings together with documents, metadata and IDs, and provides collections, persistence, metadata filtering, full-text and sparse retrieval, CRUD operations and client-server or hosted deployment options. In current Chroma, single-node vector search uses HNSW, while its broader schema and cloud architecture also support other retrieval indexes such as SPANN and sparse/full-text indexes.

Therefore, FAISS is preferable when low-level index control, custom ANN algorithms, compression or GPU optimization is the priority. Chroma is preferable when building a complete RAG application that needs database-style storage, filtering, updates and operational simplicity.

| Feature         | FAISS                 | Chroma |
| --------------- | --------------------- | ------ |
| Store vectors   | ✅                     | ✅      |
| Store documents | ❌                     | ✅      |
| Store metadata  | ❌                     | ✅      |
| Collections     | ❌                     | ✅      |
| CRUD            | Limited               | ✅      |
| Filtering       | ❌ (native)            | ✅      |
| Persistence     | Basic index save/load | ✅      |
| Client APIs     | ❌                     | ✅      |
| Server mode     | ❌                     | ✅      |


Tumhare code me FAISS ke saath document aur metadata dono store ho rahe the, but crucial point ye hai:

Unhe native FAISS store nahi kar raha tha; LangChain ka FAISS wrapper store kar raha tha.

LangChain FAISS VectorStore
│
├── FAISS index
│     └── Numerical embedding vectors
│
├── InMemoryDocstore
│     └── LangChain Document objects
│         ├── page_content
│         └── metadata
│
└── index_to_docstore_id
      └── FAISS position ko Document ID se map karta hai

vector_store = FAISS(
    embedding_function=embeddings,
    index=faiss_index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

Chroma me document, metadata, ID aur embedding same database collection ke records hain:

Chroma collection

collection.add(
    ids=["chunk-1"],
    documents=["Llama 2 is a family of language models."],
    metadatas=[
        {
            "source": "llama2.pdf",
            "page": 5
        }
    ],
    embeddings=[[0.1, 0.2, 0.3]]
)

| Component        | LangChain + FAISS             | Chroma                          |
| ---------------- | ----------------------------- | ------------------------------- |
| Embeddings       | Native FAISS index            | Chroma vector index             |
| Documents        | LangChain `Docstore`          | Chroma collection               |
| Metadata         | LangChain `Document.metadata` | Chroma collection record        |
| Mapping          | `index_to_docstore_id`        | Internally managed              |
| Save             | FAISS file + pickle           | Database persistence            |
| Metadata filters | Wrapper/application handling  | Native database filtering       |
| Collections      | Not native to FAISS           | Native                          |
| CRUD             | Wrapper/index-dependent       | Native record operations        |
| Server/cloud     | Separate system required      | Supported database architecture |


https://www.trychroma.com/

In [1]:
from dotenv import load_dotenv
import os

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

C:\Users\ASUS\AppData\Local\Temp\ipykernel_41436\3536989817.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)

In [4]:
# --------------------------------------------------
# 3. Load PDF
# --------------------------------------------------
file_path = r"C:\Users\ASUS\Desktop\Gen AI Full stak\class_21_vector_DB1_using_FAISS\FAISS\data\llama2-research-paper.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()
print("Total pages:", len(pages))

Total pages: 77


In [5]:
# --------------------------------------------------
# 4. Create chunks
# --------------------------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)
chunks = text_splitter.split_documents(pages)
print("Total chunks:", len(chunks))

Total chunks: 175


In [6]:
# --------------------------------------------------
# 5. Create Chroma vector store
# --------------------------------------------------
vector_store = Chroma(
    collection_name="llama2_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_llama2",
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

In [7]:
{
    "id": "doc-101",
    "embedding": [0.12, -0.45, 0.78, ...],
    "document": "Employees receive 20 days of annual leave.",
    "metadata": {
        "source": "hr_policy.pdf",
        "page": 5,
        "department": "HR"
    }
}

{'id': 'doc-101',
 'embedding': [0.12, -0.45, 0.78, Ellipsis],
 'document': 'Employees receive 20 days of annual leave.',
 'metadata': {'source': 'hr_policy.pdf', 'page': 5, 'department': 'HR'}}

In [8]:
# --------------------------------------------------
# 6. Add documents
# --------------------------------------------------

document_ids = vector_store.add_documents(
    documents=chunks
)

print("Documents added:", len(document_ids))

print(
    "Total documents stored:",
    vector_store._collection.count()
)


Documents added: 175
Total documents stored: 175


In [9]:
# --------------------------------------------------
# 7. Create retriever
# --------------------------------------------------

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    }
)

In [10]:
# --------------------------------------------------
# 8. Test retriever
# --------------------------------------------------

query = "What is the architecture of Llama 2?"

retrieved_documents = retriever.invoke(query)

for i, document in enumerate(
    retrieved_documents,
    start=1
):
    print(f"\n--- Retrieved document {i} ---")
    print(document.page_content[:500])
    print("Metadata:", document.metadata)



--- Retrieved document 1 ---
guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of this paper describes our pretraining methodology (Section 2), fine-tuning methodology
(Section 3), approach to model safety (Section 4), key observations and insights (Section 5), relevant related
work (Section 6), and conclusions (Section 7).
‡https://ai.meta.com/resources/models-and-libraries/llama/
§We are delayi
Metadata: {'producer': 'pdfTeX-1.40.25', 'trapped': '/False', 'keywords': '', 'creator': 'LaTeX with hyperref', 'title': '', 'subject': '', 'total_pages': 77, 'page_label': '4', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'author': '', 'moddate': '2023-07-20T00:30:36+00:00', 'source': 'C:\\Users\\ASUS\\Desktop\\Gen AI Full stak\\class_21_vector_DB1_using_FAISS\\FAISS\\data\\llama2-research-paper.pdf',

In [13]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

# Load variables from .env
load_dotenv()

# Get Groq API key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Create Groq model
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [15]:






# --------------------------------------------------
# 9. Prompt
# --------------------------------------------------

prompt = ChatPromptTemplate.from_template(
    """
    You are a question-answering assistant.

    Answer the question only from the provided context.

    If the context does not contain the answer, say:
    "I do not have enough information in the provided document."

    Context:
    {context}

    Question:
    {question}

    Answer:
    """
)


# --------------------------------------------------
# 10. Format documents
# --------------------------------------------------

def format_docs(docs):
    return "\n\n".join(
        f"""
        Source: {doc.metadata.get("source")}
        Page: {doc.metadata.get("page")}

        {doc.page_content}
        """
        for doc in docs
    )


# --------------------------------------------------
# 11. LLM
# --------------------------------------------------

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=GROQ_API_KEY
)


# --------------------------------------------------
# 12. RAG chain
# --------------------------------------------------

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | model
    | StrOutputParser()
)


# --------------------------------------------------
# 13. Ask question
# --------------------------------------------------

answer = rag_chain.invoke(
    "What is the architecture of Llama 2?"
)

print("\nFinal answer:\n")
print(answer)


Final answer:

Llama 2 is an **auto‑regressive language model that uses an optimized transformer architecture**. The tuned versions additionally employ supervised fine‑tuning (SFT) and reinforcement learning with human feedback (RLHF) to align the model to human preferences for helpfulness and safety.


In [16]:
from langchain_chroma import Chroma

loaded_vector_store = Chroma(
    collection_name="llama2_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_db_llama2"
)

In [17]:
loaded_retriever = loaded_vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

In [18]:
docs = loaded_retriever.invoke(
    "What is Llama 2?"
)

for doc in docs:
    print(doc.page_content[:500])

guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of this paper describes our pretraining methodology (Section 2), fine-tuning methodology
(Section 3), approach to model safety (Section 4), key observations and insights (Section 5), relevant related
work (Section 6), and conclusions (Section 7).
‡https://ai.meta.com/resources/models-and-libraries/llama/
§We are delayi
A.7 Model Card
Table 52 presents a model card (Mitchell et al., 2018; Anil et al., 2023) that summarizes details of the models.
Model Details
Model DevelopersMeta AI
Variations Llama 2comes in a range of parameter sizes—7B, 13B, and 70B—as well as
pretrained and fine-tuned variations.
Input Models input text only.
Output Models generate text only.
Model ArchitectureLlama 2isanauto-regressivelanguagemodelthatusesanoptimizedtransformer
architecture. The tuned versions use supervised fine-tuning (

In [20]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5,
        "filter": {
            "page": 10
        }
    }
)

In [21]:
results = vector_store.similarity_search(
    query="What is reinforcement learning?",
    k=5,
    filter={
        "page": 10
    }
)

# pusing into chroma cloud


In [22]:
import os
import chromadb

from dotenv import load_dotenv
from langchain_chroma import Chroma

load_dotenv()

client = chromadb.CloudClient(
    api_key=os.getenv("CHROMA_API_KEY"),
    tenant=os.getenv("CHROMA_TENANT"),
    database=os.getenv("CHROMA_DATABASE")
)

vector_store = Chroma(
    client=client,
    collection_name="llama2_collection",
    embedding_function=embeddings
)

print("Connected to Chroma Cloud!")

Connected to Chroma Cloud!


In [23]:
ids = vector_store.add_documents(chunks)

print(f"Uploaded {len(ids)} chunks")

Uploaded 175 chunks


In [24]:
print("Total records:", vector_store._collection.count())

Total records: 175


In [25]:
docs = vector_store.similarity_search(
    "What is the architecture of Llama 2?",
    k=3
)

for doc in docs:
    print(doc.page_content)
    print("-----")

guide¶ and code examples‖ to facilitate the safe deployment ofLlama 2 and Llama 2-Chat. More details of
our responsible release strategy can be found in Section 5.3.
The remainder of this paper describes our pretraining methodology (Section 2), fine-tuning methodology
(Section 3), approach to model safety (Section 4), key observations and insights (Section 5), relevant related
work (Section 6), and conclusions (Section 7).
‡https://ai.meta.com/resources/models-and-libraries/llama/
§We are delaying the release of the 34B model due to a lack of time to sufficiently red team.
¶https://ai.meta.com/llama
‖https://github.com/facebookresearch/llama
4
-----
A.7 Model Card
Table 52 presents a model card (Mitchell et al., 2018; Anil et al., 2023) that summarizes details of the models.
Model Details
Model DevelopersMeta AI
Variations Llama 2comes in a range of parameter sizes—7B, 13B, and 70B—as well as
pretrained and fine-tuned variations.
Input Models input text only.
Output Models generate tex